# XJTU fixed-anchor SOH forecasts

Compare SOH curves forecast once at cycle 50 or 100 for the 2C and 3C protocols. Measured SOH uses a completed discharge at the anchor. Charge-based SOH uses the charge-derived anchor and charge data through that cycle.

Select the project Python kernel and run all cells. The notebook requires NumPy and reads the saved CSV exports from notebooks 01, 06, 07, 08 and 09. CSV folders and matching ZIP archives are supported. The notebook runs on one CPU thread. At cycle 100, the 3C revision uses the voltage–capacity curve features from notebook 09 when they are available.

The notebook writes a Markdown report, group and per-battery metrics, paired trajectory rows, projected RUL at each anchor, and a ZIP archive under reports. The revised settings came from exploratory comparisons on these same XJTU cells. The results describe this cohort and are intended for verification on additional batteries.

In [1]:
import os

for variable in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
                 "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[variable] = "1"

from collections import defaultdict
from contextlib import contextmanager
from csv import DictReader, DictWriter
from dataclasses import dataclass
from io import TextIOWrapper
from pathlib import Path
from statistics import median
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np

NB01 = "lifeguard_soh_results"
NB06 = "lifeguard_rul_charge_features_results"
NB07 = "lifeguard_digital_twin_validation_results"
NB08 = "lifeguard_fixed_anchor_soh_results"
NB09 = "lifeguard_eol_timing_results"
OUTPUT_NAME = "lifeguard_fixed_soh_forecast_results"
ANCHORS = (50, 100)
MODES = ("charge", "measured")
MODE_LABELS = {"charge": "Charge-based", "measured": "Measured"}
MODELS = ("dashboard_fixed", "previous_fixed", "revised_fixed")
EOL_SOH = 80.0
EOL_TOLERANCE = 1e-9  # Account for roundoff when blending curves at the threshold.
NOMINAL_CAPACITY_AH = 2.0
LOOKAHEAD = 600
GUIDE_GRID = np.linspace(0.0, 1.0, 101)
STEPS = np.arange(LOOKAHEAD + 1)
TIME_RAMP = np.clip((STEPS - 10) / 10, 0, 1)

cwd = Path.cwd()
project = next((folder for folder in (cwd, *cwd.parents)
                if (folder / "README_LifeGuard_RL.md").is_file()
                and (folder / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open the notebook inside the LifeGuard-RL project.")
reports = project / "reports"
output = reports / OUTPUT_NAME


@contextmanager
def report_rows(folder, filename):
    path = reports / folder / filename
    archive = reports / f"{folder}.zip"
    if path.is_file():
        with path.open(encoding="utf-8", newline="") as source:
            yield DictReader(source)
        return

    if not archive.is_file():
        raise FileNotFoundError(f"Expected {path} or {archive}.")
    with ZipFile(archive) as bundle:
        members = [name for name in bundle.namelist() if Path(name).name == filename]
        if len(members) != 1:
            raise ValueError(f"Expected one {filename} in {archive.name}.")
        with bundle.open(members[0]) as data, TextIOWrapper(data, encoding="utf-8", newline="") as source:
            yield DictReader(source)

## Load the saved out-of-fold predictions and features

Use the saved notebook 07 splits. The dashboard comparison uses notebook 07 SOH-dependent fade and notebook 08 late-life calibration for Charge-based SOH at cycle 100. All comparisons use the same observed cycle keys. Verified 80% EOL is used as a training target for other cells in the fold and as an evaluation boundary for the held-out cell.

The feature window uses the latest ten cycles through the anchor and an earlier ten-cycle window ending 15 cycles before it. The duration regression for 2C at cycle 100 uses cycles 91–99 for Measured SOH and 91–100 for Charge-based SOH.

In [2]:
@dataclass(frozen=True)
class Start:
    rate: int
    fold: int
    soh: float
    rul: int


def load_reference():
    with report_rows(NB07, "eol_audit.csv") as rows:
        audit = {row["cell_id"]: row for row in rows}
    ends = {cell: int(row["observed_eol_cycle"]) for cell, row in audit.items()
            if row["event_status"] == "observed"}
    last_observed = {cell: int(row["last_observed_cycle"]) for cell, row in audit.items()}

    with report_rows(NB07, "cell_splits.csv") as rows:
        splits = {(int(row["fold"]), row["role"], row["cell_id"]) for row in rows}
    folds = sorted({fold for fold, _, _ in splits})
    test_fold = {cell: fold for fold, role, cell in splits if role == "test"}
    expected = {(fold, "test" if test_fold.get(cell) == fold else "train", cell)
                for fold in folds for cell in audit}
    if len(folds) != 5 or test_fold.keys() != audit.keys() or splits != expected:
        raise ValueError("Notebook 07 cell splits do not describe five disjoint test groups.")

    starts = {}
    with report_rows(NB07, "rollout_starts.csv") as rows:
        for row in rows:
            if row["model"] != "soh_dependent":
                continue
            cell = row["cell_id"]
            start = Start(int(row["charge_rate_c"]), int(row["fold"]),
                          float(row["initial_soh_pct"]), int(float(row["predicted_rul_cycles"])))
            if (start.fold, "test", cell) not in splits or start.rate != int(audit[cell]["charge_rate_c"]):
                raise ValueError(f"Notebook 07 start disagrees with the split for {cell}.")
            starts[int(row["landmark_cycle"]), cell, row["initializer"]] = start
    if starts.keys() != {(a, cell, mode) for a in ANCHORS for cell in audit for mode in MODES}:
        raise ValueError("Notebook 07 starts are incomplete.")

    measured = defaultdict(dict)
    state_curves = defaultdict(dict)
    with report_rows(NB07, "soh_trajectories.csv") as rows:
        for row in rows:
            if row["model"] != "soh_dependent":
                continue
            anchor, cell, mode = int(row["landmark_cycle"]), row["cell_id"], row["initializer"]
            cycle, observed = int(row["cycle"]), float(row["observed_soh_pct"])
            if measured[anchor, cell].setdefault(cycle, observed) != observed:
                raise ValueError(f"Observed SOH differs by initializer for {cell}, cycle {cycle}.")
            state_curves[anchor, cell, mode][cycle] = float(row["predicted_soh_pct"])

    late_life_curves = defaultdict(dict)
    with report_rows(NB08, "soh_trajectories.csv") as rows:
        for row in rows:
            if (int(row["landmark_cycle"]), row["initializer"], row["model"]) != (
                    100, "charge", "late_life_calibrated"):
                continue
            cell, cycle = row["cell_id"], int(row["cycle"])
            if (int(row["fold"]) != test_fold[cell]
                    or float(row["observed_soh_pct"]) != measured[100, cell].get(cycle)):
                raise ValueError(f"Notebook 08 comparison is misaligned for {cell}, cycle {cycle}.")
            late_life_curves[cell][cycle] = float(row["predicted_soh_pct"])

    for key in starts:
        anchor, cell, mode = key
        labels = measured[anchor, cell]
        endpoint = ends.get(cell, last_observed[cell])
        if (not labels or min(labels) != anchor or max(labels) != endpoint
                or state_curves[key].keys() != labels.keys()):
            raise ValueError(f"Notebook 07 scoring cycles are misaligned for {key}.")
        if cell in ends and not set(range(endpoint - 19, endpoint + 1)).issubset(labels):
            raise ValueError(f"Expected 20 labeled EOL cycles for {cell}.")
        if (anchor, mode) == (100, "charge") and late_life_curves[cell].keys() != labels.keys():
            raise ValueError(f"Notebook 08 scoring cycles are misaligned for {cell}.")

    calibration = defaultdict(list)
    with report_rows(NB07, "transition_calibration.csv") as rows:
        for row in rows:
            if row["used"].lower() == "true":
                key = int(row["fold"]), int(row["charge_rate_c"])
                calibration[key].append((float(row["bin_bottom_soh_pct"]) + 1,
                                         float(row["model_fade_pp_per_cycle"])))
    return ends, test_fold, starts, measured, state_curves, late_life_curves, calibration


def load_features(ends, test_fold):
    priors = defaultdict(dict)
    with report_rows(NB06, "rul_predictions.csv") as rows:
        for row in rows:
            cell, anchor = row["cell_id"], int(row["landmark_cycle"])
            if anchor in ANCHORS:
                if int(row["fold"]) != test_fold[cell]:
                    raise ValueError(f"Notebook 06 held-out RUL fold differs for {cell}.")
                priors[anchor, cell][row["model"]] = float(row["predicted_rul_cycles"])
    with report_rows(NB06, "cell_splits.csv") as rows:
        tests = {(int(row["landmark_cycle"]), int(row["fold"]), row["cell_id"]) for row in rows
                 if row["role"] == "test" and int(row["landmark_cycle"]) in ANCHORS}
    if tests != {(anchor, test_fold[cell], cell) for anchor in ANCHORS for cell in ends}:
        raise ValueError("Notebook 06 held-out batteries do not match Notebook 07.")

    history = defaultdict(dict)
    with report_rows(NB01, "cycle_quality.csv") as rows:
        for row in rows:
            cycle = int(row["cycle"])
            if cycle <= max(ANCHORS) and row["status"] == "ok" and row["soh_pct"]:
                history[row["cell_id"], "measured"][cycle] = float(row["soh_pct"])
    with report_rows(NB01, "cycle_features.csv") as rows:
        for row in rows:
            cycle = int(row["cycle"])
            if cycle <= max(ANCHORS) and row["prefix_charge_ah"]:
                history[row["cell_id"], "charge"][cycle] = (
                    100 * float(row["prefix_charge_ah"]) / NOMINAL_CAPACITY_AH)

    folder = reports / NB09
    has_curves = (folder / "curve_features.csv").is_file() or folder.with_suffix(".zip").is_file()
    curve_features = {}
    if has_curves:
        with report_rows(NB09, "curve_features.csv") as rows:
            curve_features = {row["cell_id"]: row for row in rows}
    return priors, history, curve_features

## Generate each curve at its anchor

Keep the initial fold-specific SOH-dependent fade and use normalized training-battery curves to project later fade. The curve shape spans zero damage at the anchor to full damage at 80% SOH. The scenario blend responds to predicted SOH and elapsed cycles.

The 2C revision adjusts the spread of plausible durations. The 3C revision keeps individual training curves at cycle 50 and uses pre-anchor curve change to choose more similar training curves at cycle 100. Each output covers a fixed 600-cycle grid. The held-out battery's end cycle is applied later during scoring.

In [3]:
def state_rollout(start, calibration):
    midpoints, fade = zip(*sorted(calibration[start.fold, start.rate]))
    curve = [start.soh]
    for _ in range(LOOKAHEAD):
        level = curve[-1]
        curve.append(max(EOL_SOH, level - np.interp(level, midpoints, fade)))
    return np.array(curve)


def training_damage(labels, anchor, endpoint):
    cycles = np.array(sorted(labels))
    values = np.array([labels[cycle] for cycle in cycles])
    if values[0] <= EOL_SOH or endpoint <= anchor:
        raise ValueError("Training curves must start above 80% SOH before EOL.")
    interpolated = np.interp(anchor + GUIDE_GRID * (endpoint - anchor), cycles, values)
    damage = (values[0] - interpolated) / (values[0] - EOL_SOH)
    damage = np.maximum.accumulate(np.clip(damage, 0, 1 - 1e-9))
    damage[0], damage[-1] = 0.0, 1.0
    return damage


def window_median(history, first, last):
    values = [history[cycle] for cycle in range(first, last + 1) if cycle in history]
    if not values:
        raise ValueError(f"No valid history at cycles {first}–{last}.")
    return median(values)


def early_features(history, anchor):
    recent = window_median(history, anchor - 9, anchor)
    past = window_median(history, anchor - 24, anchor - 15)
    return recent, (past - recent) / 15


def center_duration(anchor, cell, mode, start, training, durations, history, priors):
    if start.rate == 3:
        timing = priors[anchor, cell]
        secondary = ("ridge_3c_recent_signals_and_change" if anchor == 50
                     else "ridge_charge_history_pooled")
        return (2 * start.rul + timing["ridge_3c_recent_signals"] + timing[secondary]) / 4
    if anchor == 50:
        return float(np.median(durations))

    last = anchor - (mode == "measured")
    levels = np.array([window_median(history[other, mode], anchor - 9, last)
                       for other in training])
    query = window_median(history[cell, mode], anchor - 9, last)
    mean, scale = levels.mean(), levels.std() or 1.0
    normalized = (levels - mean) / scale
    slope = np.sum(normalized * (durations - durations.mean())) / (np.sum(normalized**2) + 3)
    return durations.mean() + (query - mean) / scale * slope


def stretch_curve(reference, duration, damage, warmup):
    duration = max(5.0, duration)
    warmup = max(0, min(warmup, int(duration // 2)))
    damage_at_warmup = np.interp(warmup / duration, GUIDE_GRID, damage)
    remaining = (1 - np.interp(STEPS / duration, GUIDE_GRID, damage)) / (1 - damage_at_warmup)
    tail = EOL_SOH + (reference[warmup] - EOL_SOH) * remaining
    return np.where(STEPS <= warmup, reference, np.maximum(EOL_SOH, tail))


def scenario_mix(reference, reference_rul, duration, damage, width, weights, warmup):
    durations = (max(5.0, duration - width), duration, duration + width) if width else (duration,)
    warmup = min(warmup, reference_rul // 2, int(min(durations) // 2))
    if not width:
        return stretch_curve(reference, duration, damage, warmup)
    return sum(weight * stretch_curve(reference, length, damage, warmup)
               for weight, length in zip(weights, durations))


def blend_late(central, faster, activation):
    weight = TIME_RAMP * np.clip((activation - central) / 4, 0, 1)
    return np.minimum.accumulate(central + weight * (faster - central))


def curve_change_distances(features, query, training, mode):
    segments = ("charge",) if mode == "charge" else ("charge", "discharge")
    values = []
    for cell in (query, *training):
        row = features.get(cell, {})
        sample = []
        for segment in segments:
            field = f"{segment}_curve_log_variance"
            if (not row.get(field) or int(row.get(f"{field}_early_count") or 0) < 5
                    or int(row.get(f"{field}_recent_count") or 0) < 5):
                return None
            sample.append(float(row[field]))
        values.append(sample)
    values = np.array(values)
    if not np.isfinite(values).all():
        return None
    return np.sum(((values[1:] - values[0]) / 1.5) ** 2, axis=1)


def forecast(anchor, cell, mode, start, training, shapes, durations, reference,
             history, priors, curve_features):
    duration = center_duration(anchor, cell, mode, start, training, durations, history, priors)
    duration_weights = 1 / (1 + ((durations - duration) / 45)**2)
    population = (np.median(shapes, axis=0) if start.rate == 2
                  else np.average(shapes, axis=0, weights=duration_weights))
    query = np.array(early_features(history[cell, mode], anchor))
    neighbors = np.array([early_features(history[other, mode], anchor) for other in training])
    early_distance = np.sum(((neighbors - query) / [1.7, 0.03]) ** 2, axis=1)
    similarity = 1 / (1 + (0.7 if start.rate == 2 else 1.3) * early_distance)
    shape_weights = similarity * duration_weights
    conditional = np.average(shapes, axis=0, weights=shape_weights)
    previous_shape = (population + conditional) / 2
    previous_duration = duration
    if anchor == 100:
        previous_duration += 0.3 * (np.average(durations, weights=similarity) - duration)

    warmup = 50 if (start.rate, anchor) == (3, 100) else 25
    central_width, fast_width, activation = {
        (2, 50): (0, 30, 85), (2, 100): (25, 30, 86),
        (3, 50): (40, 50, 92), (3, 100): (25, 50, 91),
    }[start.rate, anchor]
    central_weights = (0.25, 0.5, 0.25)
    fast_weights = (0.45, 0.35, 0.2)

    def scenarios(length, shape, spread=central_width, fast_spread=fast_width,
                  faster_weights=fast_weights):
        central = scenario_mix(reference, start.rul, length, shape, spread, central_weights, warmup)
        faster = scenario_mix(reference, start.rul, length, shape, fast_spread, faster_weights, warmup)
        return central, faster

    previous = blend_late(*scenarios(previous_duration, previous_shape), activation)
    if start.rate == 2 and anchor == 50:
        revised = blend_late(*scenarios(previous_duration, previous_shape, fast_spread=38), activation)
        return previous, revised, "wider_fast_scenarios"

    if start.rate == 2 and anchor == 100:
        population_tail = blend_late(
            *scenarios(duration, population, faster_weights=(0.35, 0.45, 0.2)), activation)
        tail_weight = np.clip((84 - population_tail) / 2, 0, 1)
        previous = np.minimum.accumulate(population_tail + tail_weight * (previous - population_tail))
        if mode == "measured":
            return previous, previous, "previous_measured"
        alternative = blend_late(*scenarios(previous_duration, previous_shape, 12, 22), activation)
        weight = 0.75 * np.clip((previous - 83) / 3, 0, 1)
        revised = np.minimum.accumulate(previous + weight * (alternative - previous))
        return previous, revised, "narrower_charge_scenarios"

    if anchor == 100:
        curve_distance = curve_change_distances(curve_features, cell, training, mode)
        if curve_distance is None:
            return previous, previous, "fallback_missing_curve_features"
        match = 1 / (1 + 1.3 * (early_distance + 0.5 * curve_distance))
        length = duration + 0.4 * (np.average(durations, weights=match) - duration)
        shape = np.average(shapes, axis=0, weights=match * duration_weights)
        revised = blend_late(*scenarios(length, shape), activation)
        return previous, revised, "curve_change_similarity"

    # Retain the relationship between curve shape and lifetime for each training battery.
    mean_life = np.average(durations, weights=shape_weights)
    weighted_scenarios = np.zeros((2, LOOKAHEAD + 1))
    for weight, shape, life in zip(shape_weights / shape_weights.sum(), shapes, durations):
        length = previous_duration + 0.15 * (life - mean_life)
        weighted_scenarios += weight * np.array(scenarios(length, (population + shape) / 2))
    individual = blend_late(*weighted_scenarios, activation)

    fast_duration = max(5, previous_duration - central_width)
    slow_duration = previous_duration + central_width
    spread_warmup = min(15, start.rul // 2, int(fast_duration // 2))
    spread = abs(stretch_curve(reference, fast_duration, previous_shape, spread_warmup)
                 - stretch_curve(reference, slow_duration, previous_shape, spread_warmup))
    correction = (0.25 * np.clip((85 - individual) / 3, -1, 1)
                  * np.clip((95 - individual) / 4, 0, 1) * np.clip(spread / 6, 0, 1) * TIME_RAMP)
    revised = np.minimum.accumulate(np.maximum(EOL_SOH, individual + correction))
    return previous, revised, "individual_3c_curves"

## Score the whole curve and the last 20 cycles

The whole-curve score starts at anchor + 1 and ends at the verified EOL or last observed cycle. The late-life window is EOL − 19 through EOL inclusive. The right-censored battery contributes whole-curve and available horizon scores. A group MAE averages per-battery MAEs equally. The worst error is the largest single-cycle absolute error over the group. Horizon scores use the exact anchor + 10 and anchor + 50 cycles where observed.

In [4]:
def point_error(cycles, errors, mask, prefix):
    matching = np.flatnonzero(mask)
    if not matching.size:
        return {f"{prefix}_points": 0, f"{prefix}_mae_pp": None,
                f"{prefix}_max_pp": None, f"{prefix}_worst_cycle": None}
    worst_index = matching[np.argmax(errors[matching])]
    return {f"{prefix}_points": int(matching.size), f"{prefix}_mae_pp": float(errors[matching].mean()),
            f"{prefix}_max_pp": float(errors[worst_index]), f"{prefix}_worst_cycle": int(cycles[worst_index])}


def evaluate():
    ends, test_fold, starts, measured, state_curves, late_life_curves, calibration = load_reference()
    priors, history, curve_features = load_features(ends, test_fold)
    training_shapes = {(anchor, cell): training_damage(measured[anchor, cell], anchor, endpoint)
                       for anchor in ANCHORS for cell, endpoint in ends.items()}

    cell_metrics = []
    trajectories = []
    rollout_starts = []
    for (anchor, cell, mode), start in sorted(starts.items()):
        training = sorted(other for other in ends
                          if starts[anchor, other, mode].rate == start.rate
                          and test_fold[other] != start.fold)
        if len(training) < 2:
            raise ValueError(f"Too few verified training curves for {cell}, fold {start.fold}.")

        reference = state_rollout(start, calibration)
        previous, revised, revision = forecast(
            anchor, cell, mode, start, training,
            np.array([training_shapes[anchor, other] for other in training]),
            np.array([ends[other] - anchor for other in training]),
            reference, history, priors, curve_features)
        for curve in (previous, revised):
            if (not np.isfinite(curve).all() or np.any(np.diff(curve) > 1e-9)
                    or np.any(curve < EOL_SOH - 1e-9) or not np.isclose(curve[0], start.soh)):
                raise ValueError(f"Invalid fixed trajectory for {cell}, anchor {anchor}, {mode}.")

        # Search the full prediction grid before restricting the forecast to observed cycles.
        crossings = np.flatnonzero(revised <= EOL_SOH + EOL_TOLERANCE)
        rul = int(crossings[0]) if crossings.size else None
        rollout_starts.append({
            "landmark_cycle": anchor, "charge_rate_c": start.rate, "initializer": mode,
            "fold": start.fold, "cell_id": cell, "model": "revised_fixed",
            "eol_soh_pct": EOL_SOH, "max_rollout_steps": LOOKAHEAD,
            "predicted_eol_cycle": anchor + rul if rul is not None else None,
            "predicted_rul_cycles": rul,
            "forecast_status": "crossed" if rul is not None else "no_crossing_by_limit",
        })

        labels = measured[anchor, cell]
        cycles = np.array(sorted(cycle for cycle in labels if cycle > anchor))
        steps = cycles - anchor
        if not steps.size or steps[-1] > LOOKAHEAD:
            raise ValueError(f"Scoring range outside the forecast grid for {cell}, anchor {anchor}.")
        observed = np.array([labels[cycle] for cycle in cycles])
        saved_state = np.array([state_curves[anchor, cell, mode][cycle] for cycle in cycles])
        np.testing.assert_allclose(reference[steps], saved_state, rtol=0, atol=1e-8)
        dashboard_path = late_life_curves[cell] if (anchor, mode) == (100, "charge") else state_curves[anchor, cell, mode]
        dashboard = np.array([dashboard_path[cycle] for cycle in cycles])
        endpoint = ends.get(cell)
        last20 = (cycles >= endpoint - 19) & (cycles <= endpoint) if endpoint is not None else np.zeros(cycles.size, dtype=bool)
        identity = {"landmark_cycle": anchor, "charge_rate_c": start.rate,
                    "initializer": mode, "fold": start.fold, "cell_id": cell,
                    "observed_eol_cycle": endpoint}

        scored = (dashboard, previous[steps], revised[steps])
        description = ("nb08_late_life" if (anchor, mode) == (100, "charge") else "nb07_soh_dependent",
                       "previous_exploratory_model", revision)
        for model, variant, prediction in zip(MODELS, description, scored):
            errors = abs(prediction - observed)
            horizons = {f"horizon_{horizon}_abs_error_pp": float(errors[steps == horizon][0])
                        if np.any(steps == horizon) else None for horizon in (10, 50)}
            cell_metrics.append({
                **identity, "model": model, "variant": variant,
                **point_error(cycles, errors, np.ones(cycles.size, dtype=bool), "full"),
                **point_error(cycles, errors, last20, "last20"), **horizons,
            })

        for cycle, step, actual, late, fixed, old, new in zip(cycles, steps, observed, last20, *scored):
            trajectories.append({
                **identity, "cycle": int(cycle), "horizon_cycles": int(step),
                "last20": bool(late), "observed_soh_pct": float(actual),
                "dashboard_fixed_soh_pct": float(fixed), "previous_fixed_soh_pct": float(old),
                "revised_fixed_soh_pct": float(new),
            })
    return cell_metrics, trajectories, rollout_starts


def aggregate(cell_metrics):
    groups = defaultdict(list)
    for row in cell_metrics:
        groups[row["landmark_cycle"], row["charge_rate_c"], row["initializer"], row["model"]].append(row)

    result = []
    conditions = sorted({(anchor, rate, mode) for anchor, rate, mode, _ in groups})
    for anchor, rate, mode in conditions:
        for model in MODELS:
            rows = groups[anchor, rate, mode, model]
            group = {"landmark_cycle": anchor, "charge_rate_c": rate, "initializer": mode, "model": model}
            for prefix in ("full", "last20"):
                available = [row for row in rows if row[f"{prefix}_points"]]
                worst = max(available, key=lambda row: row[f"{prefix}_max_pp"])
                group.update({
                    f"{prefix}_cells": len(available),
                    f"{prefix}_points": sum(row[f"{prefix}_points"] for row in available),
                    f"{prefix}_mae_pp": float(np.mean([row[f"{prefix}_mae_pp"] for row in available])),
                    f"{prefix}_max_pp": worst[f"{prefix}_max_pp"],
                    f"{prefix}_worst_cell_id": worst["cell_id"],
                    f"{prefix}_worst_cycle": worst[f"{prefix}_worst_cycle"],
                })
            for horizon in (10, 50):
                errors = [row[f"horizon_{horizon}_abs_error_pp"] for row in rows
                          if row[f"horizon_{horizon}_abs_error_pp"] is not None]
                group[f"horizon_{horizon}_cells"] = len(errors)
                group[f"horizon_{horizon}_mae_pp"] = float(np.mean(errors)) if errors else None
            group["curve_feature_fallback_cells"] = sum(
                row["variant"] == "fallback_missing_curve_features" for row in rows)
            result.append(group)
    return result

In [5]:
def write_csv(path, rows):
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def summary(group_metrics):
    by_condition = defaultdict(dict)
    for row in group_metrics:
        by_condition[row["landmark_cycle"], row["charge_rate_c"], row["initializer"]][row["model"]] = row
    lines = [
        "# XJTU fixed-anchor SOH forecast comparison",
        "",
        "All errors are SOH percentage points. Each error column compares the saved dashboard forecast, "
        "the previous exploratory method, and the revision, in that order.", "",
        "| Anchor | Charge rate | SOH input | Whole-curve MAE | Whole-curve max | Last 20 MAE | Last 20 max |",
        "|---:|---|---|---:|---:|---:|---:|",
    ]
    for (anchor, rate, mode), group in by_condition.items():
        metrics = [" → ".join(f"{group[model][field]:.3f}" for model in MODELS)
                   for field in ("full_mae_pp", "full_max_pp", "last20_mae_pp", "last20_max_pp")]
        lines.append(f"| {anchor} | {rate}C | {MODE_LABELS[mode]} | " + " | ".join(metrics) + " |")

    lines += [
        "", "## Revised forecast diagnostics", "",
        "| Anchor | Charge rate | SOH input | Whole-curve max at | Last 20 max at | +10 MAE (n) | +50 MAE (n) |",
        "|---:|---|---|---|---|---:|---:|",
    ]
    for (anchor, rate, mode), group in by_condition.items():
        row = group["revised_fixed"]
        worst = [f"{row[f'{prefix}_worst_cell_id']} / {row[f'{prefix}_worst_cycle']}"
                 for prefix in ("full", "last20")]
        horizons = [f"{row[f'horizon_{h}_mae_pp']:.3f} ({row[f'horizon_{h}_cells']})"
                    if row[f"horizon_{h}_cells"] else "—"
                    for h in (10, 50)]
        lines.append(f"| {anchor} | {rate}C | {MODE_LABELS[mode]} | " +
                     " | ".join([*worst, *horizons]) + " |")
    lines += [
        "", "The whole-curve metric weights each battery equally. The last 20 metric covers the "
        "20 cycles ending at verified EOL. The right-censored cell contributes its available "
        "whole-curve and horizon data. Each horizon lists the number of cells with that exact cycle.",
        "",
        "The revised model's projected RUL is the offset to its first 80% SOH crossing on the "
        "full forecast grid. The crossing can occur after the last measured cycle. The rollout "
        "starts CSV records the projected EOL and RUL for each battery, anchor, and SOH input.",
        "",
        "The dashboard comparison uses notebook 07 SOH-dependent fade, plus notebook 08 "
        "late-life calibration for Charge-based SOH at cycle 100. All models are evaluated on "
        "the same cells and cycles within each row. The revision was selected on this XJTU cohort. "
        "Its benefit on additional batteries still needs validation.",
    ]
    fallbacks = sum(row["curve_feature_fallback_cells"] for row in group_metrics)
    if fallbacks:
        lines += ["", f"The previous method was used for {fallbacks} forecasts with unavailable "
                  "pre-anchor curve-change features."]
    return "\n".join(lines) + "\n"


cell_metrics, trajectories, rollout_starts = evaluate()
group_metrics = aggregate(cell_metrics)

output.mkdir(parents=True, exist_ok=True)
tables = {"metrics.csv": group_metrics, "cell_metrics.csv": cell_metrics,
          "soh_trajectories.csv": trajectories, "rollout_starts.csv": rollout_starts}
for name, rows in tables.items():
    write_csv(output / name, rows)
report_text = summary(group_metrics)
(output / "summary.md").write_text(report_text, encoding="utf-8")
with ZipFile(output.with_suffix(".zip"), "w", compression=ZIP_DEFLATED) as bundle:
    for name in ("summary.md", *tables):
        bundle.write(output / name, arcname=f"{OUTPUT_NAME}/{name}")

from IPython.display import Markdown, display

display(Markdown(report_text))
print(f"Saved {output.with_suffix('.zip')}")

# XJTU fixed-anchor SOH forecast comparison

All errors are SOH percentage points. Each error column compares the saved dashboard forecast, the previous exploratory method, and the revision, in that order.

| Anchor | Charge rate | SOH input | Whole-curve MAE | Whole-curve max | Last 20 MAE | Last 20 max |
|---:|---|---|---:|---:|---:|---:|
| 50 | 2C | Charge-based | 2.537 → 0.603 → 0.603 | 9.758 → 4.260 → 4.185 | 1.411 → 0.719 → 0.724 | 3.400 → 2.507 → 2.399 |
| 50 | 2C | Measured | 2.308 → 0.564 → 0.563 | 9.500 → 4.243 → 4.168 | 1.508 → 0.719 → 0.725 | 3.400 → 2.503 → 2.394 |
| 50 | 3C | Charge-based | 1.754 → 1.482 → 1.488 | 11.450 → 8.561 → 8.071 | 3.154 → 1.675 → 1.664 | 9.564 → 5.804 → 5.730 |
| 50 | 3C | Measured | 1.895 → 1.497 → 1.495 | 12.903 → 8.664 → 8.142 | 3.614 → 1.764 → 1.756 | 12.903 → 5.837 → 5.763 |
| 100 | 2C | Charge-based | 0.964 → 0.589 → 0.576 | 4.487 → 4.031 → 4.031 | 0.743 → 0.696 → 0.696 | 3.067 → 2.316 → 2.316 |
| 100 | 2C | Measured | 1.703 → 0.567 → 0.567 | 7.650 → 4.277 → 4.277 | 1.120 → 0.708 → 0.708 | 3.400 → 2.411 → 2.411 |
| 100 | 3C | Charge-based | 1.330 → 1.219 → 1.141 | 9.123 → 6.468 → 6.405 | 2.187 → 1.509 → 1.422 | 7.120 → 4.745 → 4.655 |
| 100 | 3C | Measured | 1.539 → 1.194 → 1.124 | 9.700 → 6.519 → 6.427 | 2.558 → 1.508 → 1.429 | 7.995 → 4.636 → 4.584 |

## Revised forecast diagnostics

| Anchor | Charge rate | SOH input | Whole-curve max at | Last 20 max at | +10 MAE (n) | +50 MAE (n) |
|---:|---|---|---|---|---:|---:|
| 50 | 2C | Charge-based | XJTU_2C_battery-8 / 372 | XJTU_2C_battery-8 / 399 | 0.310 (8) | 0.380 (8) |
| 50 | 2C | Measured | XJTU_2C_battery-8 / 372 | XJTU_2C_battery-8 / 399 | 0.245 (8) | 0.323 (8) |
| 50 | 3C | Charge-based | XJTU_3C_battery-9 / 211 | XJTU_3C_battery-11 / 130 | 0.198 (15) | 0.509 (15) |
| 50 | 3C | Measured | XJTU_3C_battery-9 / 209 | XJTU_3C_battery-11 / 130 | 0.158 (15) | 0.580 (15) |
| 100 | 2C | Charge-based | XJTU_2C_battery-8 / 375 | XJTU_2C_battery-3 / 392 | 0.192 (8) | 0.322 (8) |
| 100 | 2C | Measured | XJTU_2C_battery-8 / 373 | XJTU_2C_battery-8 / 399 | 0.134 (8) | 0.265 (8) |
| 100 | 3C | Charge-based | XJTU_3C_battery-9 / 237 | XJTU_3C_battery-14 / 146 | 0.241 (15) | 0.311 (13) |
| 100 | 3C | Measured | XJTU_3C_battery-9 / 237 | XJTU_3C_battery-14 / 146 | 0.220 (15) | 0.368 (13) |

The whole-curve metric weights each battery equally. The last 20 metric covers the 20 cycles ending at verified EOL. The right-censored cell contributes its available whole-curve and horizon data. Each horizon lists the number of cells with that exact cycle.

The revised model's projected RUL is the offset to its first 80% SOH crossing on the full forecast grid. The crossing can occur after the last measured cycle. The rollout starts CSV records the projected EOL and RUL for each battery, anchor, and SOH input.

The dashboard comparison uses notebook 07 SOH-dependent fade, plus notebook 08 late-life calibration for Charge-based SOH at cycle 100. All models are evaluated on the same cells and cycles within each row. The revision was selected on this XJTU cohort. Its benefit on additional batteries still needs validation.


Saved /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_fixed_soh_forecast_results.zip
